In [102]:
import numpy as np
import pandas as pd
import random
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
PATH = '/kaggle/input/competitions/playground-series-s5e8'

In [103]:
df = pd.read_csv(f'{PATH}/train.csv')
test_df = pd.read_csv(f'{PATH}/test.csv')
ids = test_df['id']
test_df = test_df.drop('id', axis=1)

In [104]:
x = df.drop(['id', 'y'], axis=1, inplace=False)
y = df['y']

cat_cols = x.select_dtypes(include=['object']).columns.to_list()



x_train, x_val, y_train, y_val = train_test_split(x, y, train_size=0.8, random_state=SEED)

In [105]:
def validate(pipe):
    pipe.fit(x_train, y_train)
    return roc_auc_score(y_val, pipe.predict(x_val))

In [106]:
imp = SimpleImputer(strategy='median')
scaler = StandardScaler()
pca = PCA(15)

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    remainder='passthrough'
)

In [107]:
tree_pipe = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(random_state=SEED, class_weight='balanced', max_depth=10)
)

In [108]:
forest_pipe = make_pipeline(
    imp,
    scaler,
    pca,
    RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=SEED)
)

In [109]:
svc_pipe = make_pipeline(
    imp,
    scaler,
    pca,
    SVC(kernel='rbf')
)

In [110]:
ratio = (y==0).sum() / (y==1).sum()
xgb_pipe = make_pipeline(
    preprocessor,
    imp,
    XGBClassifier(n_estimators=100, max_depth=10, scale_pos_weight=ratio)
)

In [111]:
best_pipe = xgb_pipe
validate(best_pipe)

np.float64(0.8993438933666417)

In [112]:
def generate_submission(best_pipe):
    best_pipe.fit(x, y)
    pred = best_pipe.predict_proba(test_df)[:, 1]

    submission = pd.DataFrame({
        'id': ids,
        'y': pred
    })

    submission.to_csv('submission.csv', index=False)


generate_submission(best_pipe)